In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from IPython.display import display, HTML

In [2]:
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_columns', None)
# np.set_printoptions(threshold=np.inf, linewidth=400, precision=2)

# from IPython.core.interactiveshell import InteractiveShell
# InteractiveShell.ast_node_interactivity = "all"

In [3]:
# 全局变量：保存上一帧数据，用于实时计算
prev_frame = None 

In [4]:
# ===================== 数组自动网格输出（你要的格式） =====================
def auto_print(data, title):
    print(f"\n{'='*60}")
    print(f"📌 {title} | 形状 = {data.shape}")
    vec = data[0]
    n = len(vec)
    
    if n == 84:
        rows, cols = 7, 12
    elif n == 77:
        rows, cols = 7, 11
    elif n == 72:
        rows, cols = 6, 12
    elif n == 66:
        rows, cols = 6, 11
    else:
        print(vec)
        return
    
    print(f"✅ 输出格式：{rows}行 × {cols}列")
    print("-" * (cols * 7))
    grid = vec.reshape(rows, cols)
    for i in range(rows):
        print(f"行{i+1:2d} | " + " ".join(f"{v:6.1f}" for v in grid[i]))

# 打印全部行的矩阵
def print_full_matrix(data, title, rows, cols):
    print(f"\n{'='*60}")
    print(f"📌 {title} | 全部行 | {rows}×{cols}")
    for i in range(data.shape[0]):
        print(f"\n--- 第 {i+1} 行 ---")
        grid = data[i].reshape(rows, cols)
        for r in range(rows):
            print(f"行{r+1:2d} | " + " ".join(f"{v:6.1f}" for v in grid[r]))

In [5]:
# ===================== 2. 基线扣除（首帧为基准） =====================
    # 【传感器实时调用接口】
    # 输入：current_frame = 传感器实时读取的 1×84 数据
    # 输出：基线扣除后的差值帧 1×84

# 全局变量：保存第一帧
global first_frame
first_frame = None  # 初始为空

def subtract_baseline(current_frame):
    global first_frame
    current_frame = np.array(current_frame, dtype=np.float32)
    
    # 如果是第一帧 → 保存下来
    if first_frame is None:
        first_frame = current_frame.copy()
    
    # 所有帧（包括第一帧自己）都减去第一帧
    diff_frame = current_frame - first_frame
    
    return diff_frame

In [6]:
# # ===================== 2. 减去前一帧（实时差分） =====================
# # 【传感器实时调用接口】
# # 输入：current_frame = 传感器实时读取的 1×84 数据
# # 输出：当前帧 - 前一帧 的差值帧 1×84

# # 全局变量：保存上一帧
# global prev_frame
# prev_frame = None  # 初始为空

# def subtract_baseline(current_frame):
#     global prev_frame
#     current_frame = np.array(current_frame, dtype=np.float32)
    
#     # 如果是第一帧 → 没有前一帧，差值为 0
#     if prev_frame is None:
#         diff_frame = np.zeros_like(current_frame)
#     else:
#         # 正常帧：当前帧 - 前一帧
#         diff_frame = current_frame - prev_frame
    
#     # 更新：把当前帧存为下一帧的“前一帧”
#     prev_frame = current_frame.copy()
    
#     return diff_frame

In [7]:
# ===================== 2. 计算 diff_adjacent (12*6,列差分) =====================
def compute_diff_adjacent(frame):
    frame = np.array(frame)
    diff_list = []
    skip_idx = [6,13,20,27,34,41,48,55,62,69,76]
    for i in range(83):
        if i in skip_idx:
            continue
        diff = frame[i+1] - frame[i]
        diff_list.append(diff)
    return np.array(diff_list)


In [8]:
# -------------------- 3. 计算 diff_7step (11*7,行差分) --------------------
def compute_diff_7step(frame):
    frame = np.array(frame)
    diff_list = frame[:77] - frame[7:84]
    return diff_list

In [9]:
# -------------------- 4. 计算ADC角度 --------------------
def compute_gradient_angle(x_diff, y_diff):
    """
    规则：
    1. x 第 0-6 列  →  y 第 0-6 列   正常配对
    2. x 第 7 列    →  y 第 8 列   跳1列
    3. x 第 8 列    →  y 第 9 列
    4. 以此类推
    5. 一直配对到 x_diff 的倒数6列为止
    输出长度 = 66
    """
    import numpy as np

    x = np.array(x_diff)  # (72,)
    y = np.array(y_diff)  # (77,)

    x_paired = []
    y_paired = []

    x_idx = 0
    y_idx = 0

    # 一直配对，直到 x 剩下 6 列停止
    while x_idx < len(x) - 6:
        x_paired.append(x[x_idx])

        # 每 7 次，y 跳 1 列
        if x_idx % 7 == 6:
            y_idx += 1

        y_paired.append(y[y_idx])

        x_idx += 1
        y_idx += 1

    x_paired = np.array(x_paired)
    y_paired = np.array(y_paired)

    # 计算角度
    epsilon = 1e-8
    angle = np.degrees(np.arctan2(y_paired, x_paired + epsilon))
    angle = np.where(angle < 0, angle + 360, angle)

    return angle

In [10]:
# -------------------- 5. 计算Force角度 --------------------
def compute_force_angle(Fx, Fy):
    import numpy as np
    epsilon = 1e-8
    angle = np.degrees(np.arctan2(Fy, Fx + epsilon))
    angle = angle + 360 if angle < 0 else angle
    return angle

In [11]:
# -------------------- 4. 计算diff_adjacent_7step 先列再行二次差分 --------------------


In [12]:
# ================================== 主程序 ==================================
# 1. 读取数据
df = pd.read_csv('/home/qcy/Project/data/2.PZT_tangential/weight/test/data_1.csv')
data_original = df.iloc[:, 2:86].values

force_Fx = df.iloc[:, 86].values
force_Fy = df.iloc[:, 87].values

# 2. 初始化存储数组
data_diff = np.zeros_like(data_original)          # 基线扣除结果 (N,84)
diff_adjacent_list = []                           # 列差分结果 (N,72)
diff_adjacent7_list = []                          # 行差分结果 (N,77)
ADC_angle_list = []                               # 角度结果 (N,66) 
ADC_magnitudes_list = []                          # 大小/模值结果 (N,66)
force_angle_list = []                             # Force角度结果 (N,1) 
force_magnitudes_list = []                          # 大小/模值结果 (N,66)
ADC_force_angle_list = []                         # ADC角度+Force角度结果 (N,67)   

# 3. 主循环
for i in range(data_original.shape[0]):
    # -------------------- 步骤1：基线扣除 --------------------
    current_frame = data_original[i]
    diff_frame = subtract_baseline(current_frame)
    data_diff[i] = diff_frame

    # -------------------- 步骤2：列差分 --------------------
    adj_frame_row = compute_diff_adjacent(diff_frame)
    diff_adjacent_list.append(adj_frame_row)

    # -------------------- 步骤3：行差分 --------------------
    adj_frame_col = compute_diff_7step(diff_frame)
    diff_adjacent7_list.append(adj_frame_col)

    # -------------------- 步骤4：计算ADC角度 --------------------
    angle, magnitudes = compute_gradient_angle(adj_frame_row, adj_frame_col)
    ADC_angle_list.append(angle)
    ADC_magnitudes_list.append(magnitudes)

    # -------------------- 步骤5：计算Force角度 --------------------   
    fx = force_Fx[i]
    fy = force_Fy[i]
    f_angle, force_magnitude = compute_force_angle(fx, fy)
    force_angle_list.append(f_angle)
    force_magnitudes_list.append(force_magnitude)

# 转成numpy数组
diff_adjacent = np.array(diff_adjacent_list)
diff_adjacent7 = np.array(diff_adjacent7_list)
gradient_angle = np.array(ADC_angle_list)
force_angle = np.array(force_angle_list)
force_magnitudes = np.array(force_magnitudes_list)
ADC_magnitudes = np.array(ADC_magnitudes_list)


ValueError: too many values to unpack (expected 2)

In [ ]:
# ===================== 打印输出 =====================
print(f"\n✅ data_original shape: {data_original.shape}")
print_full_matrix(data_original, "data_original", 12, 7)

In [ ]:
print(f"\n✅ data_diff 基线扣除 shape: {data_diff.shape}")
print_full_matrix(data_diff, "data_diff 基线扣除", 12, 7)

In [ ]:
print(f"✅ diff_adjacent 列差分 shape: {diff_adjacent.shape}")
print_full_matrix(diff_adjacent, "diff_adjacent 列差分", 12, 6)

In [ ]:
print(f"✅ diff_adjacent7 行差分 shape: {diff_adjacent7.shape}")
print_full_matrix(diff_adjacent7, "diff_adjacent7 行差分", 11, 7)

In [ ]:
print(f"✅ gradient_angle ADC角度 shape: {gradient_angle.shape}")
print_full_matrix(gradient_angle, "gradient_angle ADC角度", 11, 6)

In [ ]:
print(f"✅ force_angle Force角度 shape: {force_angle.shape}")
print_full_matrix(force_angle, "force_angle Force角度", 1, 1)

In [ ]:
# ===================== 4. 拼接+保存 =====================
force_angle_array = np.array(force_angle_list)
force_angle = force_angle_array.reshape(-1, 1)
ADC_force_angle = np.hstack([gradient_angle, force_angle])  # 拼接

print(f"✅ force_angle ADC+Force shape: {ADC_force_angle.shape}")

# 1. 构造列名（CH1_angle ~ CH66_angle + 最后 Force_angle）
columns = [f"CH{i}_angle" for i in range(1, 67)] + ["Force_angle"]

# 2. 转成标准二维表格（带表头）
df_result = pd.DataFrame(ADC_force_angle, columns=columns)

# 3. 控制台美观输出
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 9999)
print("✅ 带列名的完整二维表格：")
print(df_result)

# 4. 保存到你指定的路径（绝对路径）
save_path = "/home/qcy/Project/data/2.PZT_tangential/weight/pre/ADC_Force_angle_result.csv"
df_result.to_csv(save_path, index=False, float_format="%.2f")

print(f"\n✅ 文件已保存到：\n{save_path}")

In [ ]:
# ===================== ADC 11×6(再缩小一半) + Force 角度 · 两张分开 =====================
import matplotlib.pyplot as plt
import numpy as np

# 关闭多余输出
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "none"

row_idx = 2
adc_angles = ADC_force_angle[row_idx, :66]
force_angle = ADC_force_angle[row_idx, -1]

# -------------------------- 图1：ADC 11×6 更小尺寸 --------------------------
fig1, axes1 = plt.subplots(11, 6, figsize=(5, 7))  # 再小一半
axes1 = axes1.ravel()

for i in range(66):
    ax = axes1[i]
    theta = np.deg2rad(adc_angles[i])
    ax.arrow(0.5, 0.5, 0.2*np.cos(theta), 0.2*np.sin(theta),
             head_width=0.09, head_length=0.09, fc='k', ec='k')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.axis('off')

plt.tight_layout(pad=0.2)
plt.suptitle(f"ADC Angles (row {row_idx+1})", fontsize=12, y=0.99)
plt.show()

# -------------------------- 图2：Force 角度 --------------------------
fig2, ax2 = plt.subplots(1, 1, figsize=(3, 3))
theta_f = np.deg2rad(force_angle)
ax2.arrow(0.5, 0.5, 0.25*np.cos(theta_f), 0.25*np.sin(theta_f),
          head_width=0.12, head_length=0.12, fc='r', ec='r', linewidth=2)
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.set_aspect('equal')
ax2.axis('off')
plt.title(f"Force Angle (row {row_idx+1})", fontsize=12)
plt.show()

In [ ]:
# ===================== 【带ADC阈值判断】ADC 11×6 + Force 角度 · 两张图 =====================
import matplotlib.pyplot as plt
import numpy as np

# 关闭多余输出
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "none"

# ===================== 配置 =====================
row_idx = 145                # 显示第3行数据
adc_threshold = 300        # ADC < 300 通道不显示

# 1. 取出数据
adc_angles = ADC_force_angle[row_idx, :66]   # ADC角度 66路
adc_raw = data_original[row_idx, :66]        # 原始ADC数据（必须是66列）
force_angle = ADC_force_angle[row_idx, -1]    # 力角度

# ===================== 图1：ADC 11行×6列（小于500不显示箭头） =====================
fig1, axes1 = plt.subplots(11, 6, figsize=(5, 7))  # 合适大小
axes1 = axes1.ravel()

for i in range(66):
    ax = axes1[i]
    
    # ========== 关键判断：ADC < 500 不画箭头 ==========
    if adc_raw[i] < adc_threshold:
        ax.axis('off')
        continue  # 跳过，不画箭头
    
    # 正常画箭头
    theta = np.deg2rad(adc_angles[i])
    ax.arrow(0.5, 0.5, 0.2 * np.cos(theta), 0.2 * np.sin(theta),
             head_width=0.09, head_length=0.09, fc='k', ec='k')
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.axis('off')

plt.tight_layout(pad=0.2)
plt.suptitle(f"ADC Angles (ADC < {adc_threshold} 隐藏)", fontsize=12, y=0.99)
plt.show()

# ===================== 图2：Force 角度（不变） =====================
fig2, ax2 = plt.subplots(1, 1, figsize=(3, 3))
theta_f = np.deg2rad(force_angle)
ax2.arrow(0.5, 0.5, 0.25 * np.cos(theta_f), 0.25 * np.sin(theta_f),
          head_width=0.12, head_length=0.12, fc='r', ec='r', linewidth=2)
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.set_aspect('equal')
ax2.axis('off')
plt.title(f"Force Angle", fontsize=12)
plt.show()